# Vanishing and Exploding Gradients

AI & ML Foundations: Neural networks and backpropagation · *Mathematics for AI and Machine Learning* by Fuheng Wu

Gradient size layer by layer in a deep network: sigmoid vs ReLU, and how the initialization scale decides whether signals survive.

Companion code for the [interactive demo](https://math4ai.org/companion/ml/vanishing-gradients/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(11)

# The demo's defaults: depth 30, width 64, W ~ N(0, g^2 / n)
depth, n = 30, 64


def grad_profile(act, gain):
    layers = []
    for _ in range(depth):
        lin = torch.nn.Linear(n, n, bias=False)
        torch.nn.init.normal_(lin.weight, std=gain / n ** 0.5)
        layers += [lin, act()]
    net = torch.nn.Sequential(*layers)
    acts = []

    def keep(module, inputs, output):       # remember each pre-activation and its gradient
        output.retain_grad()
        acts.append(output)

    hooks = [m.register_forward_hook(keep) for m in net if isinstance(m, torch.nn.Linear)]
    x = torch.randn(1, n, requires_grad=True)
    out = net(x)
    out.backward(torch.randn_like(out) / n ** 0.5)       # a random upstream gradient
    for hk in hooks:
        hk.remove()
    g = [a.grad.norm().item() for a in acts]             # gradient at each layer's pre-activation
    return g[0], g[-1], acts[-1].std().item()


print(f"depth {depth}, width {n}")
print(" activation  gain   first-layer grad   last-layer grad   ratio      last pre-activation std")
for act, gain, label in ((torch.nn.Sigmoid, 1.0, "sigmoid"), (torch.nn.Sigmoid, 4.0, "sigmoid"),
                         (torch.nn.Tanh, 1.0, "tanh"), (torch.nn.ReLU, 1.0, "relu"),
                         (torch.nn.ReLU, 2 ** 0.5, "relu (He)"), (torch.nn.ReLU, 2.0, "relu")):
    first, last, std = grad_profile(act, gain)
    print(f" {label:10s}  {gain:4.2f}   {first:14.3e}   {last:15.3e}   {first / last:9.2e}   {std:.3e}")
# Sigmoid's slope is at most 1/4, so each layer can shrink the gradient by 4x or more.
# For ReLU, gain sqrt(2) (He initialization) keeps both signals and gradients at a steady size.

depth 30, width 64
 activation  gain   first-layer grad   last-layer grad   ratio      last pre-activation std


 sigmoid     1.00        1.106e-19         1.940e-01    5.70e-19   6.185e-01
 sigmoid     4.00        1.310e-07         1.647e-01    7.95e-07   2.527e+00
 tanh        1.00        1.603e-01         9.425e-01    1.70e-01   1.336e-01
 relu        1.00        3.909e-05         6.578e-01    5.94e-05   1.127e-04
 relu (He)   1.41        1.821e-01         7.300e-01    2.49e-01   9.725e-02
 relu        2.00        3.088e+04         7.547e-01    4.09e+04   2.776e+04
